In [11]:
"""
Artificial Neural Network for Stock Price Movement Classification
Predicts stock price direction: DOWN (0), FLAT (1), UP (2)
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.utils import class_weight

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("=" * 70)
print("ARTIFICIAL NEURAL NETWORK - STOCK PRICE MOVEMENT CLASSIFICATION")
print("=" * 70)

ARTIFICIAL NEURAL NETWORK - STOCK PRICE MOVEMENT CLASSIFICATION


In [12]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n[1/6] Loading data...")

# Load training, validation, and test datasets
train_df = pd.read_csv('train_data.csv')
val_df = pd.read_csv('val_data.csv')
test_df = pd.read_csv('test_data.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")
print(f"Test data shape: {test_df.shape}")


[1/6] Loading data...
Training data shape: (6184, 23)
Validation data shape: (1552, 23)
Test data shape: (447, 23)


In [13]:
# ============================================================================
# STEP 2: DATA PREPROCESSING
# ============================================================================
print("\n[2/6] Preprocessing data...")

# Define feature columns (exclude non-numeric and target columns)
exclude_cols = ['date', 'target', 'target_encoded', 'next_close', 'price_change_pct']
numeric_feature_cols = [col for col in train_df.columns if col not in exclude_cols + ['stock']]

print(f"Number of numeric features: {len(numeric_feature_cols)}")
print(f"Numeric features: {numeric_feature_cols}")

# One-hot encode the 'stock' column
print("\nOne-hot encoding 'stock' column...")
train_stock_encoded = pd.get_dummies(train_df['stock'], prefix='stock')
val_stock_encoded = pd.get_dummies(val_df['stock'], prefix='stock')
test_stock_encoded = pd.get_dummies(test_df['stock'], prefix='stock')

# Ensure all datasets have the same stock columns
all_stock_cols = sorted(set(train_stock_encoded.columns) | set(val_stock_encoded.columns) | set(test_stock_encoded.columns))
for col in all_stock_cols:
    if col not in train_stock_encoded.columns:
        train_stock_encoded[col] = 0
    if col not in val_stock_encoded.columns:
        val_stock_encoded[col] = 0
    if col not in test_stock_encoded.columns:
        test_stock_encoded[col] = 0

train_stock_encoded = train_stock_encoded[all_stock_cols]
val_stock_encoded = val_stock_encoded[all_stock_cols]
test_stock_encoded = test_stock_encoded[all_stock_cols]

print(f"Number of stock categories (one-hot encoded): {len(all_stock_cols)}")
print(f"Stock columns: {all_stock_cols}")

# Combine numeric features and one-hot encoded stock features
X_train = np.hstack([train_df[numeric_feature_cols].values, train_stock_encoded.values])
X_val = np.hstack([val_df[numeric_feature_cols].values, val_stock_encoded.values])
X_test = np.hstack([test_df[numeric_feature_cols].values, test_stock_encoded.values])

# Target variables
y_train = train_df['target_encoded'].values
y_val = val_df['target_encoded'].values
y_test = test_df['target_encoded'].values

print(f"\nTotal features (numeric + one-hot encoded): {X_train.shape[1]}")

# Check class distribution
print(f"\nTraining set class distribution:")
print(f"  DOWN (0): {np.sum(y_train == 0)} samples")
print(f"  FLAT (1): {np.sum(y_train == 1)} samples")
print(f"  UP (2): {np.sum(y_train == 2)} samples")

print(f"\nValidation set class distribution:")
print(f"  DOWN (0): {np.sum(y_val == 0)} samples")
print(f"  FLAT (1): {np.sum(y_val == 1)} samples")
print(f"  UP (2): {np.sum(y_val == 2)} samples")

print(f"\nTest set class distribution:")
print(f"  DOWN (0): {np.sum(y_test == 0)} samples")
print(f"  FLAT (1): {np.sum(y_test == 1)} samples")
print(f"  UP (2): {np.sum(y_test == 2)} samples")

# Standardize features (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeatures standardized using StandardScaler")
print(f"Training features shape: {X_train_scaled.shape}")
print(f"Validation features shape: {X_val_scaled.shape}")
print(f"Test features shape: {X_test_scaled.shape}")

# Calculate balanced class weights
print("\nCalculating balanced class weights...")
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print(f"Class weights:")
for class_idx, weight in class_weight_dict.items():
    print(f"  Class {class_idx}: {weight:.4f}")


[2/6] Preprocessing data...
Number of numeric features: 17
Numeric features: ['open', 'high', 'low', 'close', 'volume', 'positive_sent', 'negative_sent', 'neutral_sent', 'sent_score', 'has_disclosure', 'sma_20', 'ema_12', 'ema_26', 'macd', 'signal_line', 'macd_histogram', 'rsi']

One-hot encoding 'stock' column...
Number of stock categories (one-hot encoded): 10
Stock columns: ['stock_AP', 'stock_AREIT', 'stock_CNVRG', 'stock_DMC', 'stock_JFC', 'stock_MBT', 'stock_SCC', 'stock_SM', 'stock_SMPH', 'stock_TEL']

Total features (numeric + one-hot encoded): 27

Training set class distribution:
  DOWN (0): 1398 samples
  FLAT (1): 3392 samples
  UP (2): 1394 samples

Validation set class distribution:
  DOWN (0): 310 samples
  FLAT (1): 952 samples
  UP (2): 290 samples

Test set class distribution:
  DOWN (0): 83 samples
  FLAT (1): 269 samples
  UP (2): 95 samples

Features standardized using StandardScaler
Training features shape: (6184, 27)
Validation features shape: (1552, 27)
Test fea

In [14]:
# ============================================================================
# STEP 3: HYPERPARAMETER TUNING
# ============================================================================
print("\n[3/6] Setting up hyperparameter tuning...")

# Define model builder function with tunable hyperparameters
def build_model(hp):
    model = keras.Sequential()
    
    # Tune the number of layers and units
    num_layers = hp.Int('num_layers', min_value=1, max_value=4, step=1)
    
    for i in range(num_layers):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
            activation='relu',
            name=f'hidden_layer_{i+1}'
        ))
        
        # Tune dropout rate
        if hp.Boolean(f'dropout_{i}'):
            model.add(layers.Dropout(
                rate=hp.Float(f'dropout_rate_{i}', min_value=0.1, max_value=0.5, step=0.1)
            ))
    
    # Output layer
    model.add(layers.Dense(3, activation='softmax', name='output_layer'))
    
    # Tune learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-4, 5e-4, 1e-3, 5e-3])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Initialize Keras Tuner
print("\nInitializing RandomSearch tuner...")
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='stock_ann',
    overwrite=True
)

print("\nTuner configuration:")
print(f"  Search algorithm: RandomSearch")
print(f"  Objective: val_accuracy (maximize)")
print(f"  Max trials: 20")
print(f"  Executions per trial: 1")

# Display search space
print("\nSearch space summary:")
tuner.search_space_summary()


[3/6] Setting up hyperparameter tuning...

Initializing RandomSearch tuner...

Tuner configuration:
  Search algorithm: RandomSearch
  Objective: val_accuracy (maximize)
  Max trials: 20
  Executions per trial: 1

Search space summary:
Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 4, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
dropout_0 (Boolean)
{'default': False, 'conditions': []}
learning_rate (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.0005, 0.001, 0.005], 'ordered': True}


In [15]:
# ============================================================================
# STEP 4: RUN HYPERPARAMETER SEARCH
# ============================================================================
print("\n[4/6] Running hyperparameter search...")
print("-" * 70)

# Early stopping callback
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Run the search
tuner.search(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    class_weight=class_weight_dict,
    verbose=0
)

print("-" * 70)
print("Hyperparameter search completed!")

# Get best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n" + "=" * 70)
print("BEST HYPERPARAMETERS FOUND")
print("=" * 70)
print(f"Number of layers: {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"Layer {i+1} units: {best_hps.get(f'units_{i}')}")
    if best_hps.get(f'dropout_{i}'):
        print(f"Layer {i+1} dropout rate: {best_hps.get(f'dropout_rate_{i}')}")
print(f"Learning rate: {best_hps.get('learning_rate')}")


[4/6] Running hyperparameter search...
----------------------------------------------------------------------
----------------------------------------------------------------------
Hyperparameter search completed!

BEST HYPERPARAMETERS FOUND
Number of layers: 2
Layer 1 units: 224
Layer 1 dropout rate: 0.1
Layer 2 units: 32
Learning rate: 0.005


In [16]:
# ============================================================================
# STEP 5: TRAIN FINAL MODEL WITH BEST HYPERPARAMETERS
# ============================================================================
print("\n[5/6] Training final model with best hyperparameters...")

# Build model with best hyperparameters
model = tuner.hypermodel.build(best_hps)

print("\nOptimal Model Architecture:")
model.summary()

# Train the final model with more epochs
print("\nTraining final model...")
print("-" * 70)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    class_weight=class_weight_dict,
    verbose=1
)

print("-" * 70)
print("Training completed!")


[5/6] Training final model with best hyperparameters...

Optimal Model Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training final model...
----------------------------------------------------------------------
Epoch 1/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3419 - loss: 1.1307 - val_accuracy: 0.4710 - val_loss: 1.0812
Epoch 2/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 656us/step - accuracy: 0.4391 - loss: 1.1054 - val_accuracy: 0.4562 - val_loss: 1.0932
Epoch 3/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 648us/step - accuracy: 0.4323 - loss: 1.1047 - val_accuracy: 0.4916 - val_loss: 1.0688
Epoch 4/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step - accuracy: 0.4459 - loss: 1.1047 - val_accuracy: 0.4710 - val_loss: 1.0826
Epoch 5/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step - accuracy: 0.4421 - loss: 1.1028 - val_accuracy: 0.4446 - val_loss: 1.0918
Epoch 6/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 650us/step - accuracy: 0.4420 - loss: 1.1021 - val_accuracy: 0.4362 - val_loss: 1.0951
Epoch 7/100
194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step - accuracy: 0.4406 - loss: 1.1000 - val_accuracy: 0.4716 - val_lo

In [17]:
# ============================================================================
# STEP 6: EVALUATE MODEL
# ============================================================================
print("\n[6/6] Evaluating model on test set...")

# Make predictions
y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

print("\n" + "=" * 70)
print("MODEL EVALUATION RESULTS")
print("=" * 70)

print(f"\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Test Balanced Accuracy: {test_balanced_accuracy:.4f} ({test_balanced_accuracy * 100:.2f}%)")

print("\nClassification Report:")
print("-" * 70)
target_names = ['DOWN (0)', 'FLAT (1)', 'UP (2)']
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

print("\nConfusion Matrix:")
print("-" * 70)
cm = confusion_matrix(y_test, y_pred)
print("              Predicted")
print("              DOWN  FLAT  UP")
print(f"Actual DOWN   {cm[0][0]:4d}  {cm[0][1]:4d}  {cm[0][2]:4d}")
print(f"       FLAT   {cm[1][0]:4d}  {cm[1][1]:4d}  {cm[1][2]:4d}")
print(f"       UP     {cm[2][0]:4d}  {cm[2][1]:4d}  {cm[2][2]:4d}")

# Training history summary
print("\n" + "=" * 70)
print("TRAINING HISTORY SUMMARY")
print("=" * 70)
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Final training loss: {history.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")


[6/6] Evaluating model on test set...

MODEL EVALUATION RESULTS

Test Accuracy: 0.5459 (54.59%)
Test Balanced Accuracy: 0.4177 (41.77%)

Classification Report:
----------------------------------------------------------------------
              precision    recall  f1-score   support

    DOWN (0)       0.27      0.31      0.29        83
    FLAT (1)       0.68      0.74      0.71       269
      UP (2)       0.32      0.20      0.25        95

    accuracy                           0.55       447
   macro avg       0.42      0.42      0.42       447
weighted avg       0.53      0.55      0.53       447


Confusion Matrix:
----------------------------------------------------------------------
              Predicted
              DOWN  FLAT  UP
Actual DOWN     26    44    13
       FLAT     42   199    28
       UP       27    49    19

TRAINING HISTORY SUMMARY
Final training accuracy: 0.4791
Final validation accuracy: 0.4504
Final training loss: 1.0707
Final validation loss: 1.1022


In [18]:
# ============================================================================
# PER-STOCK PERFORMANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 70)
print("PER-STOCK PERFORMANCE ANALYSIS")
print("=" * 70)

# Get stock names from test data
test_stocks = test_df['stock'].values

# Create a dataframe with predictions and actual values
results_df = pd.DataFrame({
    'stock': test_stocks,
    'actual': y_test,
    'predicted': y_pred
})

# Calculate per-stock metrics
stock_performance = []
unique_stocks = sorted(results_df['stock'].unique())

print(f"\nAnalyzing performance for {len(unique_stocks)} stocks...\n")

from sklearn.metrics import precision_recall_fscore_support

for stock in unique_stocks:
    stock_data = results_df[results_df['stock'] == stock]
    stock_actual = stock_data['actual'].values
    stock_pred = stock_data['predicted'].values
    
    # Calculate metrics
    stock_acc = accuracy_score(stock_actual, stock_pred)
    
    # Calculate precision, recall, and F1 (macro average)
    precision, recall, f1, _ = precision_recall_fscore_support(
        stock_actual, 
        stock_pred, 
        average='macro',
        zero_division=0
    )
    
    # Count samples
    n_samples = len(stock_data)
    
    # Class distribution for this stock
    n_down = np.sum(stock_actual == 0)
    n_flat = np.sum(stock_actual == 1)
    n_up = np.sum(stock_actual == 2)
    
    stock_performance.append({
        'Stock': stock,
        'Samples': n_samples,
        'Accuracy': stock_acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'DOWN': n_down,
        'FLAT': n_flat,
        'UP': n_up
    })

# Create performance dataframe
perf_df = pd.DataFrame(stock_performance)

# Sort by accuracy descending
perf_df = perf_df.sort_values('Accuracy', ascending=False)

# Display per-stock performance table
print("Per-Stock Performance Summary (sorted by accuracy):")
print("-" * 120)
print(f"{'Stock':<10} {'Samples':>7} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1-Score':>9} "
      f"{'DOWN':>5} {'FLAT':>5} {'UP':>4}")
print("-" * 120)

for _, row in perf_df.iterrows():
    print(f"{row['Stock']:<10} {row['Samples']:>7} {row['Accuracy']:>9.4f} {row['Precision']:>10.4f} "
          f"{row['Recall']:>8.4f} {row['F1-Score']:>9.4f} {row['DOWN']:>5.0f} {row['FLAT']:>5.0f} {row['UP']:>4.0f}")

print("-" * 120)

# Summary statistics
print("\nPer-Stock Performance Statistics:")
print(f"  Mean Accuracy: {perf_df['Accuracy'].mean():.4f}")
print(f"  Std Accuracy: {perf_df['Accuracy'].std():.4f}")
print(f"  Min Accuracy: {perf_df['Accuracy'].min():.4f} ({perf_df.loc[perf_df['Accuracy'].idxmin(), 'Stock']})")
print(f"  Max Accuracy: {perf_df['Accuracy'].max():.4f} ({perf_df.loc[perf_df['Accuracy'].idxmax(), 'Stock']})")

# Identify best and worst performing stocks
best_stock = perf_df.iloc[0]['Stock']
worst_stock = perf_df.iloc[-1]['Stock']

print(f"\n  Best performing stock: {best_stock} (Accuracy: {perf_df.iloc[0]['Accuracy']:.4f})")
print(f"  Worst performing stock: {worst_stock} (Accuracy: {perf_df.iloc[-1]['Accuracy']:.4f})")

# Show stocks above and below overall accuracy
above_avg = perf_df[perf_df['Accuracy'] > test_accuracy]
below_avg = perf_df[perf_df['Accuracy'] < test_accuracy]

print(f"\n  Stocks performing above overall accuracy ({test_accuracy:.4f}): {len(above_avg)}")
print(f"  Stocks performing below overall accuracy ({test_accuracy:.4f}): {len(below_avg)}")

print("\n" + "=" * 70)
print("IMPLEMENTATION COMPLETE")
print("=" * 70)


PER-STOCK PERFORMANCE ANALYSIS

Analyzing performance for 10 stocks...

Per-Stock Performance Summary (sorted by accuracy):
------------------------------------------------------------------------------------------------------------------------
Stock      Samples  Accuracy  Precision   Recall  F1-Score  DOWN  FLAT   UP
------------------------------------------------------------------------------------------------------------------------
AREIT           38    0.7895     0.2632   0.3333    0.2941     5    30    3
AP              41    0.7805     0.2667   0.3232    0.2922     5    33    3
SM              46    0.6739     0.3706   0.4345    0.4000     7    32    7
JFC             60    0.6167     0.4986   0.4402    0.4396     9    39   12
SMPH            56    0.5714     0.4603   0.4511    0.4504    12    35    9
SCC             40    0.4750     0.2500   0.3256    0.2535     7    20   13
DMC             39    0.4103     0.3935   0.3657    0.3601    10    17   12
TEL             46    0.3